In [1]:
import os, sys, subprocess
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    for name in ("GIT_URL", "GITHUB_TOKEN"):
        try:
            os.environ[name] = sec.get_secret(name)
        except Exception:
            pass
except Exception:
    pass

REPO = Path("/kaggle/working/filament-segmentation-2026")
url = os.environ.get("GIT_URL", "")
if url and not (REPO / "run.py").exists():
    token = os.environ.get("GITHUB_TOKEN", "")
    if token and url.startswith("https://"):
        url = url.replace("https://", f"https://{token}@", 1)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(REPO)])

root = REPO if (REPO / "run.py").exists() else next(p for p in Path("/kaggle/input").rglob("run.py")).parent
sys.path.insert(0, str(root))
os.chdir(root)
print(root)

!pip -q install transformers accelerate safetensors
from src.data import TRAIN_ANN, TRAIN_IMAGES
assert TRAIN_ANN.exists() and TRAIN_IMAGES.exists(), TRAIN_ANN

Cloning into '/kaggle/working/filament-segmentation-2026'...


/kaggle/working/filament-segmentation-2026


In [2]:
!python run.py train --size 1024 --epochs 8

preprocessor_config.json: 100%|████████████████| 537/537 [00:00<00:00, 2.27MB/s]
/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)
config.json: 79.8kB [00:00, 75.9MB/s]
model.safetensors: 100%|█████████████████████| 190M/190M [00:02<00:00, 84.9MB/s]
Loading weights: 100%|█| 566/566 [00:00<00:00, 1479.19it/s, Materializing param=
Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                    | Status   |                                                                                      
-----------------------+----------+--------------------------------------------------------------------------------------
class_predictor.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([81, 256]) vs model

In [3]:
%cd /kaggle/working/filament-segmentation-2026
!ls -lh /kaggle/working/outputs/m2f/last.pt
!python run.py infer


/kaggle/working
-rw-r--r-- 1 root root 182M Sep  4 13:27 /kaggle/working/outputs/m2f/last.pt
/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)
Loading weights: 100%|█| 566/566 [00:00<00:00, 1436.80it/s, Materializing param=
Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                    | Status   |                                                                                      
-----------------------+----------+--------------------------------------------------------------------------------------
class_predictor.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([81, 256]) vs model:torch.Size([2, 256])
class_predictor.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([81]